# SuperLender Loan Default Prediction

Predicts `Good_Bad_flag` (1 = good/repaid, 0 = bad/defaulted) using three datasets:
- **trainperf / testperf** — the loan to predict
- **traindemographics / testdemographics** — customer demographic info
- **trainprevloans / testprevloans** — all prior loan history

**Model:** Gradient Boosting Classifier  
**Metric:** Accuracy (minimise % incorrectly predicted)

## 1. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.impute import SimpleImputer
import matplotlib.pyplot as plt

print('Libraries loaded successfully')

## 2. Load Data

In [ ]:
train_perf = pd.read_csv('trainperf.csv')
train_demo = pd.read_csv('traindemographics.csv')
train_prev = pd.read_csv('trainprevloans.csv')

test_perf  = pd.read_csv('testperf.csv')
test_demo  = pd.read_csv('testdemographics.csv')
test_prev  = pd.read_csv('testprevloans.csv')

# Deduplicate demographics (keep first occurrence per customer)
train_demo = train_demo.drop_duplicates('customerid')
test_demo  = test_demo.drop_duplicates('customerid')

# Encode target: Good=1, Bad=0
train_perf['target'] = (train_perf['good_bad_flag'].str.lower() == 'good').astype(int)

print('Train perf shape:', train_perf.shape)
print('Test  perf shape:', test_perf.shape)
print('Target distribution:')
print(train_perf['target'].value_counts())

## 3. Feature Engineering

### 3a. Previous Loans Aggregates

For each customer, aggregate all prior loan behaviour into summary statistics that capture repayment patterns (the primary signal for willingness to pay).

In [ ]:
def prev_feats(prev):
    """Aggregate previous loan history per customer."""
    prev = prev.copy()
    for c in ['approveddate', 'closeddate', 'firstduedate', 'firstrepaiddate']:
        prev[c] = pd.to_datetime(prev[c], errors='coerce')

    prev['days_to_close'] = (prev['closeddate'] - prev['approveddate']).dt.days
    prev['repay_ratio']   = prev['totaldue'] / prev['loanamount'].replace(0, np.nan)
    prev['overdue_days']  = prev['days_to_close'] - prev['termdays']
    prev['repay_delay']   = (prev['firstrepaiddate'] - prev['firstduedate']).dt.days
    prev['early']         = (prev['repay_delay'] < 0).astype(float)
    prev['late']          = (prev['repay_delay'] > 3).astype(float)

    return prev.groupby('customerid').agg(
        prev_num_loans       = ('systemloanid', 'count'),
        prev_avg_loanamt     = ('loanamount',   'mean'),
        prev_max_loanamt     = ('loanamount',   'max'),
        prev_avg_termdays    = ('termdays',     'mean'),
        prev_avg_repay_ratio = ('repay_ratio',  'mean'),
        prev_avg_overdue     = ('overdue_days', 'mean'),
        prev_max_overdue     = ('overdue_days', 'max'),
        prev_early_rate      = ('early',        'mean'),
        prev_late_rate       = ('late',         'mean'),
        prev_avg_delay       = ('repay_delay',  'mean'),
    ).reset_index()

train_pf = prev_feats(train_prev)
test_pf  = prev_feats(test_prev)
print('Prev loan feature shape (train):', train_pf.shape)

### 3b. Demographics Features

In [ ]:
def demo_feats(train_demo, test_demo):
    """Build and encode demographic features. Fit encoder on combined data to avoid unseen labels."""
    for df in [train_demo, test_demo]:
        df['birthdate'] = pd.to_datetime(df['birthdate'], errors='coerce')
        df['age'] = (pd.Timestamp('2017-08-01') - df['birthdate']).dt.days / 365.25

    cat_cols = ['bank_account_type', 'bank_name_clients', 'employment_status_clients']
    for col in cat_cols:
        for df in [train_demo, test_demo]:
            df[col] = df[col].fillna('Unknown')
        le = LabelEncoder()
        combined = pd.concat([train_demo[col], test_demo[col]], ignore_index=True)
        le.fit(combined)
        train_demo[col + '_enc'] = le.transform(train_demo[col])
        test_demo[col  + '_enc'] = le.transform(test_demo[col])

    keep = ['customerid', 'age', 'longitude_gps', 'latitude_gps'] + [c+'_enc' for c in cat_cols]
    return train_demo[keep], test_demo[keep]

td, ed = demo_feats(train_demo, test_demo)
print('Demo feature shape (train):', td.shape)

### 3c. Current Loan Features

In [ ]:
def perf_feats(perf):
    """Extract features from the loan being predicted."""
    p = perf.copy()
    p['approveddate'] = pd.to_datetime(p['approveddate'], errors='coerce')
    p['creationdate'] = pd.to_datetime(p['creationdate'], errors='coerce')
    p['apply_lag']    = (p['approveddate'] - p['creationdate']).dt.total_seconds() / 3600
    p['interest_rate']= (p['totaldue'] / p['loanamount'].replace(0, np.nan)) - 1
    p['is_referred']  = p['referredby'].notna().astype(int)
    return p[['customerid', 'loannumber', 'loanamount', 'totaldue', 'termdays',
               'apply_lag', 'interest_rate', 'is_referred']]

tp = perf_feats(train_perf)
ep = perf_feats(test_perf)
print('Perf feature shape (train):', tp.shape)

## 4. Merge All Features

In [ ]:
train = tp.merge(td, on='customerid', how='left').merge(train_pf, on='customerid', how='left')
test  = ep.merge(ed, on='customerid', how='left').merge(test_pf,  on='customerid', how='left')

# Customers with no prior loans → fill aggregates with 0
prev_cols = [c for c in train.columns if c.startswith('prev_')]
train[prev_cols] = train[prev_cols].fillna(0)
test[prev_cols]  = test[prev_cols].fillna(0)

train['target'] = train_perf['target'].values

feat_cols = [c for c in train.columns if c not in ['customerid', 'target']]
print('Total features:', len(feat_cols))
print('Train shape:', train.shape, ' | Test shape:', test.shape)

## 5. Model Training & Cross-Validation

In [ ]:
X  = train[feat_cols]
y  = train['target']
Xt = test[feat_cols]

# Impute remaining NaNs with column median
imp   = SimpleImputer(strategy='median')
X_imp  = imp.fit_transform(X)
Xt_imp = imp.transform(Xt)

gb = GradientBoostingClassifier(
    n_estimators=400,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    min_samples_leaf=20,
    random_state=42
)

cv     = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(gb, X_imp, y, cv=cv, scoring='accuracy')

print(f'CV Accuracy:  {scores.mean():.4f} ± {scores.std():.4f}')
print(f'CV Error rate: {1 - scores.mean():.4f}')
print(f'Per-fold scores: {np.round(scores, 4)}')

## 6. Feature Importance

In [ ]:
gb.fit(X_imp, y)

fi = pd.Series(gb.feature_importances_, index=feat_cols).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 7))
fi.tail(15).plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Top 15 Feature Importances (Gradient Boosting)', fontsize=13)
ax.set_xlabel('Importance')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=120)
plt.show()
print(fi[::-1].head(15))

## 7. Generate Submission

In [ ]:
preds = gb.predict(Xt_imp)

sub = pd.DataFrame({'customerid': test['customerid'], 'Good_Bad_flag': preds})
sub.to_csv('submission.csv', index=False)

print('Submission saved: submission.csv')
print('Prediction distribution:')
print(sub['Good_Bad_flag'].value_counts())
print(sub.head(10))